# Multiple region selection

This notebook aims to detail, explain and test the region selection and the minimal go-through filter.

## Load required files and functions
### Loading parameters

In [18]:
import geopandas.geoseries
import shapely.geometry.base
%load_ext autoreload
%autoreload 2

import pandas as pd
import track_builder as tb
import os
import geopandas as gpd

data = r"D:\Stockage\ASTD"
parquet_path = "../data/"

year = 2019
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# Periods for load_periods
periods = {
    2019: months,
    2020: months
}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Loading functions
Load raw csv files and save dataframe as parquet file - (it's faster to load parquet files)\
If parquet already exists, load the parquet file as a dataframe

In [31]:
def load_data(parquet_file, source, year, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        data =  pd.read_parquet(parquet_file)
        print(f"Loaded data {parquet_file}, parameters ignored")
    else:
        data = tb.load_astd_monthly(base_path=source, year=year, **kwargs)
        data.to_parquet(parquet_file)
        print(f"Loaded data {parquet_file} from {source}")

    return data

def load_tracks(parquet_file, data, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        tracks =  pd.read_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file}, parameters ignored")
    else:
        tracks = tb.build_ship_tracks(data, **kwargs)
        tracks.to_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file} from {data}")

    return tracks

def load_periods(parquet_file, source, periods, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        period =  pd.read_parquet(parquet_file)
        print(f"Loaded period {parquet_file}, parameters ignored")
    else:
        period = tb.load_astd_periods(base_path=source, periods=periods, **kwargs)
        period.to_parquet(parquet_file)
        print(f"Loaded period {parquet_file} from {source}")

    return period

def to_box(coordinates : tuple, crs="EPSG:4326"):
    # Convert coordinates in box (handles date line crossing)
    min_lon, max_lon, min_lat, max_lat = coordinates

    # Filter Longitude (Handle Date Line crossing)
    if min_lon <= max_lon:
        # Standard case (e.g. Canada, Norway)
        geom = box(min_lon, min_lat, max_lon, max_lat)
        return gpd.GeoDataFrame(geometry=[geom], crs=crs)
    else:
        # Date Line Crossing (e.g. Russia: 50 -> -168)
        # Logic: We want longitudes > 50 OR longitudes < -168
        geom1 = box(min_lon, min_lat, 180, max_lat)
        geom2 = box(-180, min_lat, max_lon, max_lat)

        return gpd.GeoDataFrame(geometry=[geom1, geom2], crs=crs)

### Import all segments of selected year

Optional: \
Import segments first and last day to build tracks\
optional because first and last day are automatically recovered in algo, this could potentially make the process faster

In [20]:
all_data = load_data(parquet_file = f'all_segments{year}.parquet', source = data, year = year, months = months, remove_nan_rows="default", usecols="default", progress=True)
all_data.sample(5)

# Optional
# spe_track = load_data(parquet_file = f'spefirstlast{year}.parquet', source = data, year = year, months = months, remove_nan_rows="default", usecols="default", sampling=[0, -1], progress=True)
# spe_track.sample(5)

Loaded data ../data/all_segments2019.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
24053716,5361,2019-08-07 03:11:09+00:00,Russia,FS Ice Class 1C,Fishing vessels,5000 - 9999 GT,2339.561523,361,-5.490807,62.792000
18456071,3584,2019-06-26 14:04:30+00:00,Norway,FS Ice Class 1C,Passenger ships,10000 - 24999,2678.255371,360,9.517266,63.710701
8190897,16723,2019-04-03 19:30:33+00:00,Netherlands,FS Ice Class 1A,General cargo ships,1000 - 4999 GT,1408.366577,371,24.327652,65.497612
16071854,4332,2019-06-08 02:57:00+00:00,Netherlands,FS Ice Class 1A,General cargo ships,10000 - 24999,674.344788,124,30.467600,71.038620
5474591,2798,2019-03-12 04:27:49+00:00,Finland,FS Ice Class II,General cargo ships,1000 - 4999 GT,0.000000,1080,27.177334,60.530834


### Build ship tracks
Build tracks between each months' segments with last day of previous month and first day of next month

In [21]:
tracks = load_tracks(parquet_file = f'tracks{year}.parquet', data = all_data)
tracks.sample(5)

Loaded tracks ../data/tracks2019.parquet, parameters ignored


,month,segment_id,track_id
3699,2019-04,2813,2312
5795,2019-06,2128,3185
6793,2019-06,3478,3658
8566,2019-07,10588,4472
3976,2019-04,3552,2400


## Build tracks position with area of interest
`tb.build_light_multi_track_data()`
- **region** : filter tracks within region. Takes a list of region or a region; or ShapeFile/GeoJSON/Shapely Geometry defining area of interest
- **minimal_region** : filter tracks going through at least a minimal selection of region (part of previous **region** argument). Takes a list of region/area, a region/area or an
integer (minimal number of region)

> **Warning** : Point stride is automatically set to 1 to avoid removing data when region argument is passed


### Example : Only regions

In [22]:
# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "russia", "norway", "usa"], minimal_region=['russia', 'norway'],
                                               preprocess_positions=True)
display(build_tracks)
print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\579578352.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)


computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'canada'): Keeping 2 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'iceland'): Keeping 13 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'russia'): Keeping 6 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'norway'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'usa'): Keeping 1 tracks.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  track_table: The table mapping tracks to segments and months.


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,3273,2019-07-10 18:03:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2015.969727,362,-179.734711,65.031532,2019-07,4293,russia
1,3273,2019-07-10 18:09:25+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,20356.927734,3605,-179.704056,65.044151,2019-07,4293,russia
2,3273,2019-07-10 19:09:30+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,10412.908203,1793,-179.395233,65.172211,2019-07,4293,russia
3,3273,2019-07-10 19:39:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2200.736328,370,-179.232254,65.235771,2019-07,4293,russia
4,3273,2019-07-10 19:45:32+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,5301.646973,919,-179.198318,65.249451,2019-07,4293,russia
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9949,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,norway
9950,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,norway
9951,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,norway
9952,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,norway


Minimal corresponding tracks 3


In [23]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,4293,"[russia, usa, norway]"
1,5026,"[norway, russia]"
2,5274,"[russia, norway]"


### Example : Minimum number of region

In [24]:
# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "russia", "norway", "usa"], minimal_region=2 ,
                                               preprocess_positions=True)
display(build_tracks)
print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\3554994541.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)


computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'canada'): Keeping 2 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'iceland'): Keeping 13 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'russia'): Keeping 6 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'norway'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'usa'): Keeping 1 tracks.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  track_table: The table mapping tracks to segments and months.


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,2433,2019-03-01 00:02:28+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2041.190918,361,21.089647,70.898430,2019-03,1903,norway
1,2433,2019-03-01 00:08:28+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2041.525879,360,21.140417,70.906075,2019-03,1903,norway
2,2433,2019-03-01 00:14:29+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,278.866516,49,21.190943,70.913902,2019-03,1903,norway
3,2433,2019-03-01 00:15:18+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1602.995239,320,21.197899,70.914940,2019-03,1903,norway
4,2433,2019-03-01 00:20:37+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1880.493774,361,21.238394,70.920494,2019-03,1903,norway
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36236,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,norway
36237,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,norway
36238,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,norway
36239,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,norway


Minimal corresponding tracks 6


In [25]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,1903,"[norway, iceland]"
1,2046,"[norway, iceland]"
2,4293,"[russia, usa, norway]"
3,4933,"[iceland, norway]"
4,5026,"[norway, russia]"
5,5274,"[russia, norway]"


### Example : Only ShapeFile

In [26]:
# Create shape
from shapely.geometry import box

 # Norway Sea zone: (5.0, 35.0, 60.0, 82.0)
norway_zone = to_box((5.0, 35.0, 60.0, 82.0))

# Russia Sea zone: (50.0, -168.0, 65.0, 85.0)
russia_zone = to_box((50.0, -168.0, 65.0, 85.0))

# usa Sea zone: (-170.0, -140.0, 60.0, 75.0)
usa_zone = to_box((-170.0, -140.0, 60.0, 75.0))

# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=[norway_zone, russia_zone, usa_zone], minimal_region=['0','1'],
                                               preprocess_positions=True)
display(build_tracks)
print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\2839419878.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)


computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')
D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  valid_bwd = valid_fwd.shift(1).fillna(False) & is_same_ship_prev


Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '0'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '1'): Keeping 6 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '2'): Keeping 1 tracks.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  track_table: The table mapping tracks to segments and months.


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,3273,2019-07-10 18:03:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2015.969727,362,-179.734711,65.031532,2019-07,4293,1
1,3273,2019-07-10 18:09:25+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,20356.927734,3605,-179.704056,65.044151,2019-07,4293,1
2,3273,2019-07-10 19:09:30+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,10412.908203,1793,-179.395233,65.172211,2019-07,4293,1
3,3273,2019-07-10 19:39:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2200.736328,370,-179.232254,65.235771,2019-07,4293,1
4,3273,2019-07-10 19:45:32+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,5301.646973,919,-179.198318,65.249451,2019-07,4293,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9949,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,0
9950,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,0
9951,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,0
9952,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,0


Minimal corresponding tracks 3


In [27]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,4293,"[1, 2, 0]"
1,5026,"[0, 1]"
2,5274,"[1, 0]"


### Example : Minimum number of region and ShapeFile

In [32]:
# Create shape

# Norway Sea zone: (5.0, 35.0, 60.0, 82.0)
norway_zone = to_box((5.0, 35.0, 60.0, 82.0))

# Russia Sea zone: (50.0, -168.0, 65.0, 85.0)
russia_zone = to_box((50.0, -168.0, 65.0, 85.0))


# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "usa", norway_zone, russia_zone], minimal_region=2,
                                               preprocess_positions=True)
display(build_tracks)

## Correct tracks : usa doesn't appear ?? for 4293

print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\2031417246.py:11: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'canada'): Keeping 2 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'iceland'): Keeping 13 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'usa'): Keeping 1 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '0'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '1'): Keeping 6 tracks.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,2433,2019-03-01 00:02:28+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2041.190918,361,21.089647,70.898430,2019-03,1903,0
1,2433,2019-03-01 00:08:28+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,2041.525879,360,21.140417,70.906075,2019-03,1903,0
2,2433,2019-03-01 00:14:29+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,278.866516,49,21.190943,70.913902,2019-03,1903,0
3,2433,2019-03-01 00:15:18+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1602.995239,320,21.197899,70.914940,2019-03,1903,0
4,2433,2019-03-01 00:20:37+00:00,Russia,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,1880.493774,361,21.238394,70.920494,2019-03,1903,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36236,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,0
36237,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,0
36238,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,0
36239,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,0


Minimal corresponding tracks 6


In [33]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,1903,"[0, iceland]"
1,2046,"[0, iceland]"
2,4293,"[1, usa, 0]"
3,4933,"[iceland, 0]"
4,5026,"[0, 1]"
5,5274,"[1, 0]"


#### Display Shapes

In [34]:
# Show special track region coverage
# display(build_tracks[build_tracks['track_id'] == 4293]['region'].unique())

# Plot selected tracks
fig = tb.plot_ship_tracks(
    build_tracks,
    # track_ids=3478,
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=True)

# Plot shapes
from shapely.geometry import box
import plotly.graph_objects as go

regions = {
    "canada": (-141.0, -50.0, 60.0, 85.0),
    "norway": (5.0, 35.0, 60.0, 82.0),
    "russia": (50.0, -168.0, 65.0, 85.0),   # crosses dateline
    "usa": (-170.0, -140.0, 60.0, 75.0),
    "iceland": (-25.0, -12.0, 63.0, 67.0),
}

colors = {
    "canada": "red",
    "norway": "blue",
    "russia": "green",
    "usa": "purple",
    "iceland": "orange",
}

def add_box(fig, bounds, color, name):
    geom = to_box(bounds) if isinstance(bounds, tuple) else bounds
    if isinstance(geom, gpd.GeoDataFrame):
        geom = geom['geometry'].tolist()
    elif not isinstance(geom, list):
        geom = [geom]

    for g in geom:
        x, y = g.exterior.xy

        fig.add_trace(
            go.Scattermapbox(
                lon=list(x),
                lat=list(y),
                mode="lines",
                line=dict(color=color, width=2),
                name=name,
                showlegend=True
            )
        )


# Add all regions
for region, bounds in regions.items():
    add_box(fig, bounds, colors[region], region)


fig.show()

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\41178563.py:47: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\41178563.py:47: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\41178563.py:47: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\41178563.py:47: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\41178563.py:47: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/pyt

### Example : ShapeFiles and Regions

In [35]:
# Create shape

# Norway Sea zone: (5.0, 35.0, 60.0, 82.0)
norway_zone = to_box((5.0, 35.0, 60.0, 82.0))

# Russia Sea zone: (50.0, -168.0, 65.0, 85.0)
russia_zone = to_box((50.0, -168.0, 65.0, 85.0))

# usa Sea zone: (-170.0, -140.0, 60.0, 75.0)
usa_zone = to_box((-170.0, -140.0, 60.0, 75.0))

# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "norway", russia_zone, usa_zone], minimal_region=["norway", "0"],
                                               preprocess_positions=True)
display(build_tracks)

print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\1253142582.py:13: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'canada'): Keeping 2 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'iceland'): Keeping 13 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'norway'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '0'): Keeping 6 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '1'): Keeping 1 tracks.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,3273,2019-07-10 18:03:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2015.969727,362,-179.734711,65.031532,2019-07,4293,0
1,3273,2019-07-10 18:09:25+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,20356.927734,3605,-179.704056,65.044151,2019-07,4293,0
2,3273,2019-07-10 19:09:30+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,10412.908203,1793,-179.395233,65.172211,2019-07,4293,0
3,3273,2019-07-10 19:39:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2200.736328,370,-179.232254,65.235771,2019-07,4293,0
4,3273,2019-07-10 19:45:32+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,5301.646973,919,-179.198318,65.249451,2019-07,4293,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9949,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,norway
9950,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,norway
9951,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,norway
9952,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,norway


Minimal corresponding tracks 3


In [36]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,4293,"[0, 1, norway]"
1,5026,"[norway, 0]"
2,5274,"[0, norway]"


### Example : Remove overlapping regions

In [37]:
# Create shape

# Norway Sea zone: (5.0, 35.0, 60.0, 82.0)
norway_zone = to_box((5.0, 35.0, 60.0, 82.0))

# Russia Sea zone: (50.0, -168.0, 65.0, 85.0)
russia_zone = to_box((50.0, -168.0, 65.0, 85.0))

# usa Sea zone: (-170.0, -140.0, 60.0, 75.0)
usa_zone = to_box((-170.0, -140.0, 60.0, 75.0))

# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "norway", russia_zone, usa_zone], minimal_region=["norway", "0"],
                                               preprocess_positions=True, remove_overlapping=True)
display(build_tracks)

print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)

C:\Users\virtu\AppData\Local\Temp\ipykernel_5276\862967425.py:13: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:383: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:171: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Cleaning completed: 1420 'ghost' or aberrant points removed.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'canada'): Keeping 2 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'iceland'): Keeping 13 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key 'norway'): Keeping 72 tracks.
Using provided GeoDataFrame for spatial filter.
  -> Spatial Filter (key '0'): Keeping 6 tracks.
Using provided GeoDataFrame for spatial filter.
Geometry key '1' overlaps with geometry key 'canada'. Skipping region.
Geometry key '1' overlaps with geometry key '0'. Skipping region.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:215: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,3273,2019-07-10 18:03:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2015.969727,362,-179.734711,65.031532,2019-07,4293,0
1,3273,2019-07-10 18:09:25+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,20356.927734,3605,-179.704056,65.044151,2019-07,4293,0
2,3273,2019-07-10 19:09:30+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,10412.908203,1793,-179.395233,65.172211,2019-07,4293,0
3,3273,2019-07-10 19:39:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2200.736328,370,-179.232254,65.235771,2019-07,4293,0
4,3273,2019-07-10 19:45:32+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,5301.646973,919,-179.198318,65.249451,2019-07,4293,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9938,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,norway
9939,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,norway
9940,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,norway
9941,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,norway


Minimal corresponding tracks 3


In [38]:
# Show tracks
display(build_tracks.groupby('track_id')['region'].unique().reset_index())

,track_id,region
0,4293,"[0, norway]"
1,5026,"[norway, 0]"
2,5274,"[0, norway]"


## Longest track

In [39]:
def get_longest_tracks(tracks : pd.DataFrame, n_tracks : int =5):
    # Verify column name exists
    if not {'track_id', 'dist_nextpoint'}.issubset(tracks.columns):
        print("No tracks found")
        return None

    # Get the total distance for each tracks
    total_dist_ship = (
        tracks.groupby('track_id', as_index=False)['dist_nextpoint']
        .sum()
        .rename(columns={'dist_nextpoint': 'total_distance'})
        .sort_values('total_distance', ascending=False)
    )

    df_longest_tracks = total_dist_ship.head(n_tracks)
    longest_track_ids = df_longest_tracks['track_id'].to_list()

    print("Longest track segments")
    display(tracks[tracks['track_id'] == longest_track_ids[0]])

    print('Longest tracks')
    display(df_longest_tracks)

    return longest_track_ids

In [40]:
longest_track_ids = None
print("Minimal corresponding tracks", build_tracks['track_id'].nunique() if build_tracks.shape[0] > 0 else 0)
#
if build_tracks['track_id'].nunique() > 0:
    # Compute longest_tracks
    longest_track_ids = get_longest_tracks(build_tracks)


Minimal corresponding tracks 3
Longest track segments


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
5355,3271,2019-09-01 13:25:15+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,4830.845703,782,50.106907,70.026428,2019-09,5274,0
5356,3271,2019-09-01 13:38:16+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,4219.992188,661,50.233307,70.029282,2019-09,5274,0
5357,3271,2019-09-01 13:49:18+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,6793.356445,1056,50.343700,70.031952,2019-09,5274,0
5358,3271,2019-09-01 14:06:54+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,8104.631836,1276,50.521465,70.036156,2019-09,5274,0
5359,3271,2019-09-01 14:28:09+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,28270.044922,4522,50.733574,70.041252,2019-09,5274,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9938,2162,2019-12-31 06:32:38+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.462871,5,33.032372,68.958664,2019-12,5274,norway
9939,2162,2019-12-31 14:45:10+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,3.707131,362,33.032372,68.958664,2019-12,5274,norway
9940,2162,2019-12-31 17:40:13+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.386500,184,33.032387,68.958687,2019-12,5274,norway
9941,2162,2019-12-31 22:45:55+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.678558,531,33.032372,68.958664,2019-12,5274,norway


Longest tracks


,track_id,total_distance
2,5274,14113247.0
0,4293,13603924.0
1,5026,5920465.0


### Visualize longest tracks

In [41]:
# Show longest tracks
display(longest_track_ids)
display(build_tracks[build_tracks['track_id'].isin(longest_track_ids)]['region'].unique())

# Clean horizontal lines
# build_tracks = tb.core.track_helpers.mask_dateline_jumps(build_tracks)

fig = tb.plot_ship_tracks(
    build_tracks,
    track_ids=longest_track_ids,
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=True)
fig.show()

[5274, 4293, 5026]

array(['0', 'norway'], dtype=object)

In [42]:
# Visualize USA points on track 4293
display(build_tracks[build_tracks['track_id'] == 4293]['region'].unique())
usa_track = build_tracks[(build_tracks['region'] == "1") & (build_tracks['track_id'] == 4293)]

fig = tb.plot_ship_tracks(
    usa_track,
    track_ids=longest_track_ids + [4293],
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=True)
fig.show()

array(['0', 'norway'], dtype=object)